In [21]:
import pandas as pd
import re
import nltk
import json
from nltk.corpus import stopwords
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [ ]:
#english dataset preprocesing
nltk.download("stopwords")
df = pd.read_csv("engnews.csv")
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)
stop_words = set(stopwords.words("english"))
def clean_engtext(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)
df["clean_text"] = df["text"].apply(clean_engtext)
df = df[df["clean_text"].str.strip() != ""]
df = df[["clean_text", "FAKE"]]
df.to_csv("clean_english_news_dataset.csv", index=False)
#print("Clean dataset saved")
#print("Total rows:", len(df))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lrc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#Hindi dataset preprocessing
df = pd.read_csv("hindinews.csv", encoding="utf-8")
eng_stop = set(stopwords.words("english"))
hinglish_stop = set(stopwords.words("hinglish"))
stop_words = eng_stop.union(hinglish_stop)
def clean_hinditext(text):
    text = str(text)
    text = transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)
df["clean_text"] = df["text"].apply(clean_hinditext)
df = df[["clean_text", "FAKE"]]
df.to_csv("clean_hindi_news_dataset.csv", index=False, encoding="utf-8-sig")
#print("Clean dataset created successfully")

In [25]:
df1 = pd.read_csv("clean_english_news_dataset.csv")
df2 = pd.read_csv("clean_hindi_news_dataset.csv")
merged = pd.concat([df1, df2], ignore_index=True)
merged.to_csv("merged_ds.csv", index=False)
#print("done")

In [ ]:
#vectorization of merged dataset
'''df = pd.read_csv("merged_ds.csv")
X = df["clean_text"]
y = df["FAKE"]
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_vectorized = vectorizer.fit_transform(X)
#print("Vectorized shape:", X_vectorized.shape)
#print("vectorization done")'''

'df = pd.read_csv("merged_ds.csv")\nX = df["clean_text"]\ny = df["FAKE"]\nvectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))\nX_vectorized = vectorizer.fit_transform(X)\n#print("Vectorized shape:", X_vectorized.shape)\n#print("vectorization done")'

In [26]:
df = pd.read_csv("merged_ds.csv")
X = df['clean_text']  
y = df['FAKE']  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9467025303845388
Confusion Matrix:
 [[4603  271]
 [ 264 4900]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.94      0.95      4874
           1       0.95      0.95      0.95      5164

    accuracy                           0.95     10038
   macro avg       0.95      0.95      0.95     10038
weighted avg       0.95      0.95      0.95     10038



In [ ]:
with open("news_json.json", "r", encoding="utf-8") as file:
    data = json.load(file)
articles = data["articles"]
#articles is a list of dictionary
text = []
titles = []
#text is list of content of each article
for i in articles:
    temp = i["content"]
    text.append(temp)
    temp = i["title"]
    titles.append(temp)

In [34]:
def detect_lang(text):
    # Hindi characters range
    if re.search(r'[\u0900-\u097F]', text):
        return "hindi"
    return "eng"

def credibility_score(news_text):
    if news_text is None:
        return {"Probability" : 0 , "Label" : "None"} 
    else:
        language = detect_lang(news_text)
        if language=="hindi":
            clean= clean_hinditext(news_text)
        elif language=="eng":
            clean = clean_engtext(news_text)
        else:
            print("Error in determining language of the news")

    
        vec = vectorizer.transform([clean])
        fake_prob = model.predict_proba(vec)[0][1]
        ml_score = 1 - fake_prob

        if ml_score > 0.7:
            label = "LIKELY TRUE"
        elif ml_score > 0.4:
            label = "UNCERTAIN"
        else:
            label = "LIKELY FAKE"
    
        result = {"Probability" : ml_score , "Label" : label}
        return result


In [35]:
for i in range(len(titles)):
    result = credibility_score(text[i])
    z = i+1
    print(z,".\nTitle : ",titles[i],"\n")
    for key,value in result.items():
        print(key,":",value,"\n")

1 .
Title :  Morgan Stanley Limits Redemptions on Private Credit Fund - Bloomberg 

Probability : 0.8734473769281751 

Label : LIKELY TRUE 

2 .
Title :  Jack Osbourne Names Newborn Baby Girl After Ozzy - TMZ 

Probability : 0.03413512236229732 

Label : LIKELY FAKE 

3 .
Title :  Kyler Murray weighing Vikings starting job, backup options elsewhere after Cardinals release - Arizona Sports 

Probability : 0.48969946759308824 

Label : UNCERTAIN 

4 .
Title :  Trump will tap oil reserve as Iran war drives up gas prices - Axios 

Probability : 0.8618220932350896 

Label : LIKELY TRUE 

5 .
Title :  Live updates: Strait of Hormuz in crosshairs of Iran conflict as attacks on ships escalate - CNN 

Probability : 0.6686702649736813 

Label : UNCERTAIN 

6 .
Title :  Trump takes anti-Massie crusade to Kentucky in stark escalation - Axios 

Probability : 0.16564148292694092 

Label : LIKELY FAKE 

7 .
Title :  AI ‘actor’ Tilly Norwood put out the worst song I’ve ever heard - TechCrunch 

Probab

In [ ]:
'''reliable_sources = {"bbc.com","reuters.com","nytimes.com","theguardian.com"}
unreliable_sources = {"clickbaitnews.com","fakeportal.com"}
def detect_lang(text):
    # Hindi characters range
    if re.search(r'[\u0900-\u097F]', text):
        return "hindi"
    return "eng"
    
def credibility_score(news_text, source):
    language = detect_lang(news_text)
    if language=="hindi":
        clean= clean_hinditext(news_text)
    elif language=="eng":
        clean = clean_engtext(news_text)
    else:
        print("Error in determining language of the news")

    
    vec = vectorizer.transform([clean])
    fake_prob = model.predict_proba(vec)[0][1]
    ml_score = 1 - fake_prob

#1.if news from reliable source
    if source in reliable_sources:
        source_score = 1
    elif source in unreliable_sources:
        source_score = 0.1
    else:
        source_score = 0.5
#2.if news already in database
    verified_news = set(df["clean_text"])
    if clean in verified_news:
        db_score = 1
    else:
        db_score = 0.5

    similar_count = sum(clean in article for article in verified_news)

    if similar_count > 5:
        cross_score = 1
    elif similar_count > 1:
        cross_score = 0.6
    else:
        cross_score = 0.2

    final_score = (0.5 * ml_score +  0.25 * source_score + 0.15 * cross_score)
    #final_score = (0.5 * ml_score +  0.25 * source_score + 0.15 * cross_score + 0.10 * db_score)

    if final_score > 0.7:
        label = "LIKELY TRUE"
    elif final_score > 0.4:
        label = "UNCERTAIN"
    else:
        label = "LIKELY FAKE"

    return {
        "credibility_score": round(final_score,3),
        "fake_probability": round(fake_prob,3),
        "result": label}'''

'reliable_sources = {"bbc.com","reuters.com","nytimes.com","theguardian.com"}\nunreliable_sources = {"clickbaitnews.com","fakeportal.com"}\ndef detect_lang(text):\n    # Hindi characters range\n    if re.search(r\'[ऀ-ॿ]\', text):\n        return "hindi"\n    return "eng"\n\ndef credibility_score(news_text, source):\n    language = detect_lang(news_text)\n    if language=="hindi":\n        clean= clean_hinditext(news_text)\n    elif language=="eng":\n        clean = clean_engtext(news_text)\n    else:\n        print("Error in determining language of the news")\n\n\n    vec = vectorizer.transform([clean])\n    fake_prob = model.predict_proba(vec)[0][1]\n    ml_score = 1 - fake_prob\n\n#1.if news from reliable source\n    if source in reliable_sources:\n        source_score = 1\n    elif source in unreliable_sources:\n        source_score = 0.1\n    else:\n        source_score = 0.5\n#2.if news already in database\n    verified_news = set(df["clean_text"])\n    if clean in verified_news:\

In [ ]:
'''with open("engreal.txt", "r", encoding="utf-8") as f:
        news_text = f.read()
source="bbc.com"
result = credibility_score(news_text, source)
print("engreal:-")
for key,value in result.items():
    print(key," : ",value)



with open("engfake.txt", "r", encoding="utf-8") as f:
        news_text = f.read()
source="bbc.com"
result = credibility_score(news_text, source)
print("engfake:-")
for key,value in result.items():
    print(key," : ",value)



with open("hindireal.txt", "r", encoding="utf-8") as f:
        news_text = f.read()
source="bbc.com"
result = credibility_score(news_text, source)
print("hindireal:-")
for key,value in result.items():
    print(key," : ",value)



with open("hindifake.txt", "r", encoding="utf-8") as f:
        news_text = f.read()
source="xyz.com"
result = credibility_score(news_text, source)
print("hindifake:-")
for key,value in result.items():
    print(key," : ",value)'''

'with open("engreal.txt", "r", encoding="utf-8") as f:\n        news_text = f.read()\nsource="bbc.com"\nresult = credibility_score(news_text, source)\nprint("engreal:-")\nfor key,value in result.items():\n    print(key," : ",value)\n\n\n\nwith open("engfake.txt", "r", encoding="utf-8") as f:\n        news_text = f.read()\nsource="bbc.com"\nresult = credibility_score(news_text, source)\nprint("engfake:-")\nfor key,value in result.items():\n    print(key," : ",value)\n\n\n\nwith open("hindireal.txt", "r", encoding="utf-8") as f:\n        news_text = f.read()\nsource="bbc.com"\nresult = credibility_score(news_text, source)\nprint("hindireal:-")\nfor key,value in result.items():\n    print(key," : ",value)\n\n\n\nwith open("hindifake.txt", "r", encoding="utf-8") as f:\n        news_text = f.read()\nsource="xyz.com"\nresult = credibility_score(news_text, source)\nprint("hindifake:-")\nfor key,value in result.items():\n    print(key," : ",value)'